# Airline DW Cleaning Notebook

This notebook cleans airline datasets and builds curated dimension/fact tables using the current project structure.

In [ ]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

In [ ]:
# Project paths based on current notebook location
BASE_PATH = Path.cwd()
RAW_DATA_PATH = BASE_PATH / '03_Data_Raw'
CURATED_PATH = BASE_PATH / '04_Data_Cleaned' / 'curated_dw'
CURATED_PATH.mkdir(parents=True, exist_ok=True)

print('Base path:', BASE_PATH)
print('Raw data path:', RAW_DATA_PATH)
print('Curated output path:', CURATED_PATH)

In [ ]:
dataset_files = {
    'airline_satisfaction': 'airline_passenger_satisfaction.csv',
    'customer_activity': 'Customer Flight Activity.csv',
    'customer_loyalty': 'Customer Loyalty History.csv',
    'calendar': 'Calendar.csv'
}

raw_data = {}
for key, filename in dataset_files.items():
    filepath = RAW_DATA_PATH / filename
    if not filepath.exists():
        raise FileNotFoundError(f'Missing file: {filepath}')
    raw_data[key] = pd.read_csv(filepath)

{k: v.shape for k, v in raw_data.items()}

In [ ]:
def standardize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('[^a-z0-9_]', '', regex=True)
    )
    return df

cleaned_data = {k: standardize_cols(v) for k, v in raw_data.items()}
{k: list(v.columns[:8]) for k, v in cleaned_data.items()}

In [ ]:
# Deduplicate with business keys
cleaned_data['airline_satisfaction'] = cleaned_data['airline_satisfaction'].drop_duplicates(subset=['id'], keep='first')
cleaned_data['customer_activity'] = cleaned_data['customer_activity'].drop_duplicates(subset=['loyalty_number', 'year', 'month'], keep='first')
cleaned_data['customer_loyalty'] = cleaned_data['customer_loyalty'].drop_duplicates(subset=['loyalty_number'], keep='first')

{k: v.shape for k, v in cleaned_data.items()}

In [ ]:
# Surrogate keys
unique_cust = cleaned_data['customer_loyalty'][['loyalty_number']].drop_duplicates().reset_index(drop=True)
unique_cust['customer_sk'] = range(1, len(unique_cust) + 1)
customer_key_map = dict(zip(unique_cust['loyalty_number'], unique_cust['customer_sk']))

unique_dates = cleaned_data['calendar'][['date']].drop_duplicates().reset_index(drop=True)
unique_dates['date_sk'] = range(1, len(unique_dates) + 1)
date_key_map = dict(zip(unique_dates['date'], unique_dates['date_sk']))

len(customer_key_map), len(date_key_map)

In [ ]:
# Build dimensions
dim_customer = cleaned_data['customer_loyalty'].copy()
dim_customer['customer_sk'] = dim_customer['loyalty_number'].map(customer_key_map)
dim_customer = dim_customer[[
    'customer_sk', 'loyalty_number', 'country', 'province', 'city',
    'postal_code', 'gender', 'education', 'salary', 'marital_status',
    'loyalty_card', 'clv', 'enrollment_type', 'enrollment_year', 'enrollment_month'
]]
dim_customer['load_date'] = datetime.now().date()

dim_time = cleaned_data['calendar'].copy()
dim_time['date_sk'] = dim_time['date'].map(date_key_map)
dim_time = dim_time[['date_sk', 'date', 'start_of_year', 'start_of_quarter', 'start_of_month']].copy()
dim_time['load_date'] = datetime.now().date()

dim_customer.shape, dim_time.shape

In [ ]:
# Build facts
fact_sat = cleaned_data['airline_satisfaction'].copy()
fact_sat['satisfaction_sk'] = fact_sat.index + 1

fact_act = cleaned_data['customer_activity'].copy()
fact_act['activity_sk'] = fact_act.index + 1
fact_act['customer_sk'] = fact_act['loyalty_number'].map(customer_key_map)

fact_sat.shape, fact_act.shape

In [ ]:
# Validate referential integrity
fact_fks = set(fact_act['customer_sk'].dropna().unique())
dim_pks = set(dim_customer['customer_sk'].unique())
orphaned = fact_fks - dim_pks
coverage = len(fact_fks - orphaned) / len(fact_fks) * 100 if fact_fks else 100

print(f'fact_activity -> dim_customer coverage: {coverage:.2f}%')
print(f'orphaned keys: {len(orphaned)}')

In [ ]:
# Export cleaned tables
dim_customer.to_csv(CURATED_PATH / 'dim_customer_cleaned.csv', index=False)
dim_time.to_csv(CURATED_PATH / 'dim_time_cleaned.csv', index=False)
fact_sat.to_csv(CURATED_PATH / 'fact_satisfaction_cleaned.csv', index=False)
fact_act.to_csv(CURATED_PATH / 'fact_activity_cleaned.csv', index=False)

# Export metadata
dw_schema = {
    'schema_name': 'airline_dw',
    'version': '1.0',
    'created': datetime.now().isoformat(),
    'dimensions': {
        'dim_customer': {'primary_key': 'customer_sk', 'business_key': 'loyalty_number', 'rows': len(dim_customer)},
        'dim_time': {'primary_key': 'date_sk', 'business_key': 'date', 'rows': len(dim_time)}
    },
    'facts': {
        'fact_satisfaction': {'primary_key': 'satisfaction_sk', 'rows': len(fact_sat)},
        'fact_activity': {'primary_key': 'activity_sk', 'rows': len(fact_act), 'foreign_keys': {'customer_sk': 'dim_customer.customer_sk'}}
    }
}

sql_templates = {
    'join_customer_activity': """
    SELECT c.customer_sk, c.loyalty_number, c.country, c.loyalty_card, c.clv,
           fa.year, fa.month, fa.total_flights, fa.distance,
           fa.points_accumulated, fa.points_redeemed
    FROM fact_activity fa
    INNER JOIN dim_customer c ON fa.customer_sk = c.customer_sk
    ORDER BY c.loyalty_number, fa.year, fa.month
    """
}

with open(CURATED_PATH / 'dw_schema_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(dw_schema, f, indent=2, default=str)

with open(CURATED_PATH / 'dw_sql_templates.json', 'w', encoding='utf-8') as f:
    json.dump(sql_templates, f, indent=2)

audit_lines = [
    'AIRLINE DATA WAREHOUSE PREPARATION AUDIT',
    f'Executed: {datetime.now().isoformat()}',
    '=' * 80,
    f'dim_customer rows: {len(dim_customer):,}',
    f'dim_time rows: {len(dim_time):,}',
    f'fact_satisfaction rows: {len(fact_sat):,}',
    f'fact_activity rows: {len(fact_act):,}',
    f'FK coverage fact_activity -> dim_customer: {coverage:.2f}%',
]

with open(CURATED_PATH / 'data_cleaning_audit.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(audit_lines) + '\n')

print('Export completed to:', CURATED_PATH)

## Next Step

Load files from 04_Data_Cleaned/curated_dw in Power BI and build the dashboards from the setup guide.